### Tests the Marko parsing of footnotes
From [here](https://www.perplexity.ai/search/i-ve-attached-a-markdown-docum-5rfc3AkNSbOYE7_UeLn_GA) for example.

**NO AI MODEL CAN DO THIS...**

In [8]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict, Counter
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic
import re
from typing import Optional, Dict, List, Tuple
import datetime as dt

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz
import re

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# Test Markdown input
markdown_text = '''
<img src="https://r2cdn.perplexity.ai/pplx-full-logo-primary-dark%402x.png" class="logo" width="120"/>

# Describe the current state of Seattle's Burke Gilman missing link bike trail, and explain how it reached its current state.

Don't forget that this extra bit of text 

is also part of the prompt!

---
The "Missing Link" of Seattle's Burke-Gilman Trail remains incomplete as of February 2025. Despite being a priority in Seattle's Bicycle Master Plan since the 1990s, the Missing Link has faced resistance from local businesses along the proposed route and extensive litigation under the State Environmental Policy Act (SEPA), which opponents have used to delay progress[^1][^4][^9].

### Current State

- **Unfinished Section**: The Missing Link spans 1.4 miles around Salmon Bay, east of the Ballard Locks. Cyclists must navigate busy streets or alternative routes to bypass this gap[^1][^9].
- **Litigation and Delays**: Legal challenges have stalled construction repeatedly, making it one of Seattle's most prolonged infrastructure disputes[^1][^9].

### How It Reached This State

1. **Historical Background**: The Burke-Gilman Trail was initially developed on a former railroad corridor and opened in sections starting in 1978. While most of the 27-mile trail is complete, the Ballard segment has remained contentious due to its industrial setting[^5][^10].
3. **City Efforts**: Successive city administrations have allocated funds and conducted studies to close the gap. However, these efforts have been met with resistance, forcing SDOT to consider alternative routes[^7][^9].

The Missing Link remains a critical gap[^1][^7][^9].

<div style="text-align: center">⁂</div>

[^1]: https://www.youtube.com/watch?v=TKEdfRlFfnw
[^2]: https://www.theurbanist.org/2015/06/22/finding-the-burke-gilman-trails-missing-link/
[^3]: https://www.seattlebikeblog.com/2017/10/12/the-community-advised-missing-link-design-keeps-getting-better-for-everyone/
[^4]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/bgtmissinglink
[^5]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-multimodal-corridor/burke-gilman-trail-history
[^6]: https://www.traillink.com/trail/burke-gilman-trail/
[^7]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-bike-route-study
[^8]: https://www.seattlebikeblog.com/2015/03/31/cascade-publishes-very-very-long-timeline-of-work-to-build-the-burke-gilman-missing-link/
[^9]: https://www.seattletimes.com/seattle-news/transportation/is-the-burke-gilman-missing-link-in-seattle-finally-getting-built/
[^10]: https://www.seattlepi.com/seattlenews/article/History-Burke-Gilman-bicycle-walk-trail-Seattle-14369100.php
'''

# This is my version:  it works, and it's just plain python

In [ ]:
PROMPT_END_STR_PERPLEX = '---'    
BODY_SOURCES_DIVIDER_STR = '<div style="text-align: center">⁂</div>'
source_list_pattern_perplex = re.compile(r'\[\^?(?P<num>\d+)\]:\s*(?P<url>http[s]?://\S+)')

relinker = lpz.ZoteroLinkConverter()

def split_dedup_chat_text_perplex(markdown_text: str) -> Tuple[str, str, str]:
    """Splits perplexity output markdown text into prompt, response and source sections."""

    match = re.search(r'(?m)^# (?P<heading_text>.+)', markdown_text)
    if (heading_start_index := match.start('heading_text')) == -1:
        raise ValueError('Could not find prompt heading')
    
    if (prompt_end_index := markdown_text.rfind(PROMPT_END_STR_PERPLEX)) == -1:
        raise ValueError(f'Could not find prompt prompt end string "{PROMPT_END_STR_PERPLEX}"')
    
    if heading_start_index >= prompt_end_index:
        raise ValueError(f'{heading_start_index=} >= {prompt_end_index=}.  Probably missed the starting level 1 header part of the prompt."')
    
    prompt = markdown_text[heading_start_index:prompt_end_index].strip()

    body_sources_divider_index = markdown_text.rfind(BODY_SOURCES_DIVIDER_STR)

    if body_sources_divider_index == -1:
        raise ValueError('Could not find divider between AI response and sources list')

    if body_sources_divider_index <= prompt_end_index:
        raise ValueError('body_sources_divider_index <= prompt_end_index')

    sources_list = f"{markdown_text[body_sources_divider_index:]}" #.strip()

    source_citenum_url_pairs = []
    for m in re.finditer(source_list_pattern_perplex, sources_list):
        source_citenum_url_pairs.append((m.group('num'), m.group('url')))
    if len(source_citenum_url_pairs) < 1:
        print('Found no sources in file_text')
   
    citenums_to_url_source = relinker.citenums_to_urls_dedup(source_citenum_url_pairs)

    response_start_index = prompt_end_index + len(PROMPT_END_STR_PERPLEX)
    response = f"{markdown_text[response_start_index:body_sources_divider_index]}".strip()
    response_dedup = relinker.replace_body_citenums(response, citenums_to_url_source.new_num.to_dict())
    
    return prompt, response_dedup, citenums_to_url_source

prompt, response_dedup, citenums_to_url_source = split_dedup_chat_text_perplex(markdown_text)

ic(prompt, citenums_to_url_source, response_dedup);


Reading from cache.


ic| prompt: ("Describe the current state of Seattle's Burke Gilman missing link bike "
             'trail, and explain how it reached its current state.
            '
             '
            '
             "Don't forget that this extra bit of text 
            "
             '
            '
             'is also part of the prompt!')
    citenums_to_url_source:          new_num                                                url
                            orig_num                                                           
                            1              1        https://www.youtube.com/watch?v=TKEdfRlFfnw
                            2              2  https://www.theurbanist.org/2015/06/22/finding...
                            3              3  https://www.seattlebikeblog.com/2017/10/12/the...
                            4              4  https://www.seattle.gov/transportation/project...
                            5              5  https://www.seattle.gov/transportation

# Marko almost worked: but it did almost nothing.  Ordinary python does the heavy lifting

In [11]:
# gets 1st header, footnote dict, subs footnotes, gets the AI response, correctly turns source list into markdown links, gets the full prompt
# BUT - source list is not separated out (I didn't ask for that, and its easy to make)
# BUT - I'm not sure that it handles duplicated citenums
# BUT - rendered_ai_response is truncated

# import marko

# #def parse_markdown_perplex_content(markdown_text):
# # Initialize Marko with footnote extension
# markdown = marko.Markdown(extensions=['footnote'])

# # Parse the markdown
# parsed_doc = markdown.parse(markdown_text)

# # 1. Get first level 1 headline
# headline = None
# for element in parsed_doc.children:
#     if isinstance(element, marko.block.Heading) and element.level == 1:
#         headline = ''.join(refwrangle/link_perplexity_zotero.pystr(child.children) for child in element.children)
#         break

# # 2. Create footnote dictionary by parsing the footnote definitions directly
# source_citenum_url_pairs = {}
# import re

# # Use regex to find footnote definitions
# # TODO: store num/url tuples, so can fix duplicates
# footnote_pattern = r'\[\^(\d+)\]:\s*(http[s]?://\S+)'
# for match in re.finditer(footnote_pattern, markdown_text):
#     number, url = match.groups()
#     source_citenum_url_pairs[number] = url

# # 3. Replace footnotes with markdown links in the entire document (rendered_text)
# # TODO: dedup using tuples
# rendered = markdown_text
# for number, url in source_citenum_url_pairs.items():
#     rendered = rendered.replace(f'[^{number}]', f'[{number}]({url})')

# # 4. Extract full_prompt (text of heading 1 + text below it until last "---")
# prompt = None
# ai_response = None
# rendered_ai_response = None

# if headline:
#     # Find the start of the first-level heading and extract text below it until "---"
#     heading_start_index = markdown_text.find(f"# {headline}")
#     PROMPT_END_STR_PERPLEX = '---'    
#     prompt_end_index = markdown_text.rfind(PROMPT_END_STR_PERPLEX)  # Find last occurrence of divider from the top

#     if heading_start_index != -1 and prompt_end_index != -1:
#         # Extract text from just below the heading to the last divider
#         full_prompt_start_index = heading_start_index + len(f"# {headline}")
#         prompt = f"{headline}\n" + markdown_text[full_prompt_start_index:prompt_end_index].strip()

#         # Extract ai_response (text from last divider "---" to footnotes)
#         ai_response_start_index = prompt_end_index + len('---')
#         footnotes_start_index = markdown_text.find('[^')  # Footnotes typically start with "[^"
        
#         if footnotes_start_index == -1:  # If no footnotes exist, take all remaining text
#             ai_response = markdown_text[ai_response_start_index:].strip()
#         else:
#             ai_response = markdown_text[ai_response_start_index:footnotes_start_index].strip()

#         # Render ai_response (replace footnotes with links)
#         rendered_ai_response = ai_response
#         for number, url in source_citenum_url_pairs.items():
#             rendered_ai_response = rendered_ai_response.replace(f'[^{number}]', f'[{number}]({url})')

# result = {
#     'headline': headline,
#     'footnotes': source_citenum_url_pairs,
#     'rendered_text': rendered,
#     'full_prompt': prompt,
#     'ai_response': ai_response,
#     'rendered_ai_response': rendered_ai_response
# }
    
#    return result

# def parse_markdown_perplex_file(file_path):
#     markdown_text = lpz.read_markdown_file(file_path)
#     return parse_markdown_perplex_content(markdown_text)

#result = parse_markdown_perplex_file(file_path)
#result = parse_markdown_perplex_content(markdown_text)
# result


# Try MarkdownIt:  Hopeless!

In [12]:
# from markdown_it import MarkdownIt
# from mdit_py_plugins.footnote import footnote_plugin
# import re

# def process_markdown(md_text):
#     # Initialize parser with footnote plugin
#     md = MarkdownIt().use(footnote_plugin)
#     env = {}
#     tokens = md.parse(md_text, env)
    
#     results = {
#         'first_h1': None,
#         'footnotes': {},
#         'modified_text': md_text
#     }
    
#     # 1. Find first H1 heading
#     for i, token in enumerate(tokens):
#         if token.type == 'heading_open' and token.tag == 'h1':
#             if tokens[i+1].type == 'inline':
#                 results['first_h1'] = tokens[i+1].content
#                 break
#     ic(env['footnotes']['list'])
#     # 2. Extract footnotes from env
#     if 'footnotes' in env and 'list' in env['footnotes']:
#         for i, footnote in enumerate(env['footnotes']['list']):
#             # Check if footnote has content
#             if footnote['content']:
#                 # Extract the URL from the tokens within the footnote content
#                 url = footnote['content']
#                 results['footnotes'][str(i + 1)] = url  # Footnote keys start from 1

#     # 3. Replace footnote references with markdown links in the main text
#     def replace_footnotes(match):
#         fn_id = match.group(1)
#         if fn_id in results['footnotes']:
#             return f"[{fn_id}]({results['footnotes'][fn_id]})"
#         return match.group(0)

#     results['modified_text'] = re.sub(r'\[\^(\d+)\]', replace_footnotes, results['modified_text'])

#     # 4. Replace footnote definitions with markdown links at the bottom
#     def replace_footnote_definitions(match):
#         fn_id = match.group(1)
#         if fn_id in results['footnotes']:
#             url = results['footnotes'][fn_id]
#             return f"[^{fn_id}]: [{fn_id}]({url}): {url}"
#         return match.group(0)

#     results['modified_text'] = re.sub(r'\[\^(\d+)\]:\s*(.*)', replace_footnote_definitions, results['modified_text'], flags=re.MULTILINE)
    
#     return results


# # Example usage:
# markdown_text = """
# <img src="https://r2cdn.perplexity.ai/pplx-full-logo-primary-dark%402x.png" class="logo" width="120"/>

# # Describe the current state of Seattle's Burke Gilman missing link bike trail, and explain how it reached its current state.

# ---
# The "Missing Link" of Seattle's Burke-Gilman Trail remains incomplete as of February 2025. Despite being a priority in Seattle's Bicycle Master Plan since the 1990s, the Missing Link has faced resistance from local businesses along the proposed route and extensive litigation under the State Environmental Policy Act (SEPA), which opponents have used to delay progress[^1][^4][^9].

# ### Current State

# - **Unfinished Section**: The Missing Link spans 1.4 miles around Salmon Bay, east of the Ballard Locks. Cyclists must navigate busy streets or alternative routes to bypass this gap[^1][^9].
# - **Litigation and Delays**: Legal challenges have stalled construction repeatedly, making it one of Seattle's most prolonged infrastructure disputes[^1][^9].

# ### How It Reached This State

# 1. **Historical Background**: The Burke-Gilman Trail was initially developed on a former railroad corridor and opened in sections starting in 1978. While most of the 27-mile trail is complete, the Ballard segment has remained contentious due to its industrial setting[^5][^10].
# 3. **City Efforts**: Successive city administrations have allocated funds and conducted studies to close the gap. However, these efforts have been met with resistance, forcing SDOT to consider alternative routes[^7][^9].

# The Missing Link remains a critical gap[^1][^7][^9].

# <div style="text-align: center">⁂</div>

# [^1]: https://www.youtube.com/watch?v=TKEdfRlFfnw

# [^2]: https://www.theurbanist.org/2015/06/22/finding-the-burke-gilman-trails-missing-link/

# [^3]: https://www.seattlebikeblog.com/2017/10/12/the-community-advised-missing-link-design-keeps-getting-better-for-everyone/

# [^4]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/bgtmissinglink

# [^5]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-multimodal-corridor/burke-gilman-trail-history

# [^6]: https://www.traillink.com/trail/burke-gilman-trail/

# [^7]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-bike-route-study

# [^8]: https://www.seattlebikeblog.com/2015/03/31/cascade-publishes-very-very-long-timeline-of-work-to-build-the-burke-gilman-missing-link/

# [^9]: https://www.seattletimes.com/seattle-news/transportation/is-the-burke-gilman-missing-link-in-seattle-finally-getting-built/

# [^10]: https://www.seattlepi.com/seattlenews/article/History-Burke-Gilman-bicycle-walk-trail-Seattle-14369100.php
# """

# output = process_markdown(markdown_text)

# print("1. First H1:", output['first_h1'])
# print("\n2. Footnotes:", output['footnotes'])
# print("\n3. Modified Text:\n", output['modified_text'])


In [13]:
# datdir = rfw.refwrangle_test_dir / 'dat'

# # file_path_perp = datdir / "perplexity_example.md"
# # file_path_smc = datdir / "perplexity_single_prompt_savemychatbot_example.md"
# # file_path_not_md = datdir / 'obsnotecitekeys.csv'
# # file_path_no_exist = datdir / "__alsdkfjasd__.md"
# tmpdir = pl.Path(r'~/tmp')

# file_path = pl.Path(r"C:\Users\scott\tmp\describe_perplex.md")
# file_path = pl.Path(r"C:\Users\scott\tmp\Are any bears there_.md")

# markdown_text = lpz.read_markdown_file(file_path)
# result = process_markdown(markdown_text)

# Another Marko attempt: Fail

In [14]:
# from marko import Markdown
# from marko.ext.footnote import make_extension

# # Input Markdown document with footnotes
# input_markdown = """
# This is an example of a document with footnotes[^1]. You can use footnotes to provide additional context or references[^2].

# [^1]: https://example.com/footnote1
# [^2]: https://example.com/footnote2
# """

# def convert_footnotes_to_links(markdown_text):
#     # Initialize Marko with the footnote extension
#     markdown = Markdown(extensions=['footnote'])
    
#     # Parse the input Markdown into an AST
#     doc = markdown.parse(markdown_text)
    
#     # Extract footnotes and their content
#     footnotes = {}
#     for child in doc.children:
#         if hasattr(child, 'element') and child.element == 'footnote_definition':
#             # Get the footnote label and content
#             label = child.label
#             # Extract the text content from the first paragraph of the footnote
#             content = child.children[0].children[0].children
#             footnotes[label] = content
    
#     # Create a new renderer that converts footnote references to links
#     class LinkRenderer(markdown.renderer):
#         def render_footnote_reference(self, element):
#             if element.label in footnotes:
#                 return f'[{element.label}]({footnotes[element.label]})'
#             return super().render_footnote_reference(element)
    
#     # Render with our custom renderer
#     markdown.renderer = LinkRenderer
#     main_text = markdown.render(doc)
    
#     # Generate the footnote list
#     footnote_list = "\n".join(
#         f"[{label}]: {content}" for label, content in footnotes.items()
#     )
    
#     return f"{main_text}\n\n---\n\n### Footnotes\n\n{footnote_list}"

# # Convert and print the result
# output = convert_footnotes_to_links(input_markdown)
# print(output)
